In [ ]:
import os
import logging
import warnings
import joblib
import pandas as pd

logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

os.environ["CMDSTAN_LOG"] = "0"

from prophet import Prophet

MODEL_FILE = "attendance_forecasting_models.pkl"
EXCEL_FILE = "student_attendance (2).xlsx"


def get_level(value):

    if value <= 75:
        return "Low"

    elif value <= 90:
        return "Medium"

    else:
        return "High"


def create_model(history):

    data = pd.DataFrame({
        "ds": pd.to_datetime(history["ds"]),
        "y": history["y"].astype(float)
    })



    model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10
)

    model.fit(data)

    return model



def predict_next_month(model):

    future = model.make_future_dataframe(
        periods=1,
        freq="MS"
    )

    forecast = model.predict(future)

    prediction = forecast.iloc[-1]["yhat"]

    prediction = max(
        0,
        min(100, prediction)
    )

    return round(float(prediction), 2)


def load_models():

    if os.path.exists(MODEL_FILE):

        return joblib.load(MODEL_FILE)

    return {}


def save_models(models):

    joblib.dump(
        models,
        MODEL_FILE
    )


def train_existing_students():

    df = pd.read_excel(EXCEL_FILE)

    months = [
        "Jan", "Feb", "Mar", "Apr", "May",
        "Jun", "Jul", "Aug", "Sep", "Oct", "Nov"
    ]

    models = {}

    dates = pd.date_range(
        start="2026-01-01",
        periods=11,
        freq="MS"
    )

    for _, student in df.iterrows():

        history = pd.DataFrame({
            "ds": dates,
            "y": [
                float(student[month])
                for month in months
            ]
        })

        model = create_model(history)

        student_id = str(student["Student_ID"])

        models[student_id] = {
            "student_name": str(
                student["Student_Name"]
            ),
            "history": history,
            "model": model
        }

    save_models(models)

    print("All student models saved successfully.")


def predict_student(student_id):

    models = load_models()

    student_id = str(student_id)

    if student_id not in models:

        print("Student ID not found.")

        return

    student_data = models[student_id]

    prediction = predict_next_month(
        student_data["model"]
    )

    level = get_level(prediction)

    history = student_data["history"]

    next_month = (
        pd.to_datetime(history["ds"].iloc[-1])
        + pd.offsets.MonthBegin(1)
    )

    print()
    print("Student ID:", student_id)
    print(
        "Student Name:",
        student_data["student_name"]
    )
    print(
        "Predicted Month:",
        next_month.strftime("%B %Y")
    )
    print(
        "Predicted Attendance:",
        prediction,
        "%"
    )
    print(
        "Predicted Attendance Level:",
        level
    )


def add_new_student(
    student_id,
    student_name,
    attendance_values
):

    models = load_models()

    student_id = str(student_id)

    if student_id in models:

        print("Student ID already exists.")

        return

    dates = pd.date_range(
        start="2026-01-01",
        periods=len(attendance_values),
        freq="MS"
    )

    history = pd.DataFrame({
        "ds": dates,
        "y": [
            float(value)
            for value in attendance_values
        ]
    })

    model = create_model(history)

    models[student_id] = {
        "student_name": str(student_name),
        "history": history,
        "model": model
    }

    save_models(models)

    prediction = predict_next_month(model)

    level = get_level(prediction)

    next_month = (
        dates[-1]
        + pd.offsets.MonthBegin(1)
    )

    print()
    print("New student added successfully.")
    print("Student ID:", student_id)
    print("Student Name:", student_name)
    print(
        "Predicted Month:",
        next_month.strftime("%B %Y")
    )
    print(
        "Predicted Attendance:",
        prediction,
        "%"
    )
    print(
        "Predicted Attendance Level:",
        level
    )


def update_student_month(
    student_id,
    actual_attendance
):

    models = load_models()

    student_id = str(student_id)

    if student_id not in models:

        print("Student ID not found.")

        return

    student_data = models[student_id]

    history = student_data["history"].copy()

    last_date = pd.to_datetime(
        history["ds"].iloc[-1]
    )

    next_date = (
        last_date
        + pd.offsets.MonthBegin(1)
    )

    new_row = pd.DataFrame({
        "ds": [next_date],
        "y": [float(actual_attendance)]
    })

    history = pd.concat(
        [history, new_row],
        ignore_index=True
    )

    model = create_model(history)

    models[student_id]["history"] = history
    models[student_id]["model"] = model

    save_models(models)

    prediction = predict_next_month(model)

    level = get_level(prediction)

    next_month = (
        next_date
        + pd.offsets.MonthBegin(1)
    )

    print()
    print("Attendance updated successfully.")
    print("Student ID:", student_id)
    print(
        "Actual Attendance:",
        actual_attendance,
        "%"
    )
    print(
        "Next Predicted Month:",
        next_month.strftime("%B %Y")
    )
    print(
        "Predicted Attendance:",
        prediction,
        "%"
    )
    print(
        "Predicted Attendance Level:",
        level
    )


def predict_all_students():

    models = load_models()

    results = []

    for student_id, student_data in models.items():

        prediction = predict_next_month(
            student_data["model"]
        )

        level = get_level(prediction)

        history = student_data["history"]

        next_month = (
            pd.to_datetime(history["ds"].iloc[-1])
            + pd.offsets.MonthBegin(1)
        )

        results.append({
            "Student_ID": student_id,
            "Student_Name":
                student_data["student_name"],
            "Predicted_Month":
                next_month.strftime("%B %Y"),
            "Predicted_Attendance":
                prediction,
            "Predicted_Level":
                level
        })

    result = pd.DataFrame(results)

    print(result)

    result.to_excel(
        "Attendance_Forecast_Result.xlsx",
        index=False
    )

    return result


if __name__ == "__main__":

    train_existing_students()

    predict_student("STU0001")

    add_new_student(
        "STU1050",
        "Anjali",
        [
            82, 85, 84, 86, 88,
            87, 90, 89, 91, 92, 93
        ]
    )

    update_student_month(
        "STU0001",
        96.50
    )

    predict_all_students()

In [ ]:
import logging
logging.disable(logging.CRITICAL)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from prophet import Prophet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


df = pd.read_excel("student_attendance (2).xlsx")

months = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov"
]


def get_level(value):

    if value <= 75:
        return "Low"

    elif value <= 90:
        return "Medium"

    return "High"


actual_oct = []
predicted_oct = []

actual_nov = []
predicted_nov = []


for _, student in df.iterrows():

    data = pd.DataFrame({
        "ds": pd.date_range(
            start="2026-01-01",
            periods=9,
            freq="MS"
        ),
        "y": [
            student[month]
            for month in months[:9]
        ]
    })

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(data)

    future = model.make_future_dataframe(
        periods=1,
        freq="MS"
    )

    forecast = model.predict(future)

    prediction = forecast.iloc[-1]["yhat"]

    prediction = max(
        0,
        min(100, prediction)
    )

    actual_oct.append(
        float(student["Oct"])
    )

    predicted_oct.append(
        float(prediction)
    )


    data = pd.DataFrame({
        "ds": pd.date_range(
            start="2026-01-01",
            periods=10,
            freq="MS"
        ),
        "y": [
            student[month]
            for month in months[:10]
        ]
    })

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(data)

    future = model.make_future_dataframe(
        periods=1,
        freq="MS"
    )

    forecast = model.predict(future)

    prediction = forecast.iloc[-1]["yhat"]

    prediction = max(
        0,
        min(100, prediction)
    )

    actual_nov.append(
        float(student["Nov"])
    )

    predicted_nov.append(
        float(prediction)
    )


oct_mae = mean_absolute_error(
    actual_oct,
    predicted_oct
)

oct_rmse = np.sqrt(
    mean_squared_error(
        actual_oct,
        predicted_oct
    )
)

oct_r2 = r2_score(
    actual_oct,
    predicted_oct
)


nov_mae = mean_absolute_error(
    actual_nov,
    predicted_nov
)

nov_rmse = np.sqrt(
    mean_squared_error(
        actual_nov,
        predicted_nov
    )
)

nov_r2 = r2_score(
    actual_nov,
    predicted_nov
)


actual_oct_level = [
    get_level(x)
    for x in actual_oct
]

predicted_oct_level = [
    get_level(x)
    for x in predicted_oct
]


actual_nov_level = [
    get_level(x)
    for x in actual_nov
]

predicted_nov_level = [
    get_level(x)
    for x in predicted_nov
]


oct_accuracy = accuracy_score(
    actual_oct_level,
    predicted_oct_level
)

oct_precision = precision_score(
    actual_oct_level,
    predicted_oct_level,
    average="weighted",
    zero_division=0
)

oct_recall = recall_score(
    actual_oct_level,
    predicted_oct_level,
    average="weighted",
    zero_division=0
)

oct_f1 = f1_score(
    actual_oct_level,
    predicted_oct_level,
    average="weighted",
    zero_division=0
)


nov_accuracy = accuracy_score(
    actual_nov_level,
    predicted_nov_level
)

nov_precision = precision_score(
    actual_nov_level,
    predicted_nov_level,
    average="weighted",
    zero_division=0
)

nov_recall = recall_score(
    actual_nov_level,
    predicted_nov_level,
    average="weighted",
    zero_division=0
)

nov_f1 = f1_score(
    actual_nov_level,
    predicted_nov_level,
    average="weighted",
    zero_division=0
)


print("\nREGRESSION METRICS")
print("------------------------------")

print("\n9 Months -> October")
print("MAE :", round(oct_mae, 3))
print("RMSE:", round(oct_rmse, 3))
print("R2  :", round(oct_r2, 3))

print("\n10 Months -> November")
print("MAE :", round(nov_mae, 3))
print("RMSE:", round(nov_rmse, 3))
print("R2  :", round(nov_r2, 3))


print("\nCLASSIFICATION METRICS")
print("------------------------------")

print("\n9 Months -> October")
print("Accuracy :", round(oct_accuracy, 3))
print("Precision:", round(oct_precision, 3))
print("Recall   :", round(oct_recall, 3))
print("F1 Score :", round(oct_f1, 3))

print("\n10 Months -> November")
print("Accuracy :", round(nov_accuracy, 3))
print("Precision:", round(nov_precision, 3))
print("Recall   :", round(nov_recall, 3))
print("F1 Score :", round(nov_f1, 3))


print("\nSAMPLE PREDICTIONS")
print("------------------------------")

for i in range(10):

    print(
        df.iloc[i]["Student_ID"],
        "| Actual Oct:",
        round(actual_oct[i], 2),
        "| Predicted Oct:",
        round(predicted_oct[i], 2),
        "| Actual Level:",
        actual_oct_level[i],
        "| Predicted Level:",
        predicted_oct_level[i]
    )


print("\nNOVEMBER SAMPLE")

for i in range(10):

    print(
        df.iloc[i]["Student_ID"],
        "| Actual Nov:",
        round(actual_nov[i], 2),
        "| Predicted Nov:",
        round(predicted_nov[i], 2),
        "| Actual Level:",
        actual_nov_level[i],
        "| Predicted Level:",
        predicted_nov_level[i]
    )